# Bank Distress Early-Warning Model: Feature Engineering
**MSDS 696 Data Science Practicum II · Oussama Ennaciri**

Picks up from `cleaning.ipynb` and adds the one thing the panel is missing: **history**.

Every row today is a single quarter, a still frame. But the EDA finding is that banks
decline over *two years* before they cross into distress, and a still frame cannot show a
decline by construction. Petropoulos et al. (2020) feed two years of lagged observations
per feature for exactly this reason; Cole & White (2012) and Correia/Luck/Verner (2024)
both single out asset growth.

This notebook adds, for each core measure, **how much it changed over the past 4 and 8
quarters**, and nothing else. No values are altered, no rows dropped.

Output: `panel_trend.parquet`, which `modeling.ipynb` reads.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

PROCESSED = Path("..") / "data" / "processed"

panel = pd.read_parquet(PROCESSED / "panel_clean.parquet")
print(f"panel_clean: {panel.shape[0]:,} rows x {panel.shape[1]} cols")

panel_clean: 1,258,888 rows x 67 cols


## 1 · Which measures get a history

Not all 54 features, only the ones the literature treats as vitals, grouped by CAMELS
category. Adding a trend to every column would double the table for no gain: a bank's
state code does not have a trend, and the macro columns are identical for every bank in a
quarter, so their "change" would be one number repeated 19,000 times.

Two kinds of change, because the columns mean different things:

| Kind | Columns | Computed as | Reads as |
|---|---|---|---|
| **Ratios** (already %) | capital, asset quality, earnings, funding | today − a year ago | "fell 2.3 points" |
| **Dollar levels** | `ASSET`, `DEP` | today ÷ a year ago − 1 | "grew 14%" |

Subtracting dollar amounts would just measure bank size, which is why those two get a
growth rate instead.

In [2]:
# Ratios -> change in percentage points.
RATIO_TREND = [
    # capital
    "RBCRWAJ", "RBC1AAJ", "EQV",
    # asset quality
    "NPERFV", "NCLNLSR", "ORER", "NTLNLSR", "LNATRESR",
    # earnings
    "ROA", "ROE", "NIMY", "EEFFR",
    # funding / liquidity  (the SVB blind spot)
    "DEPUNA", "BROR", "LNLSDEPR", "CHBALR",
]

# Dollar levels -> growth rate.
GROWTH_TREND = ["ASSET", "DEP"]

# 4 quarters = 1 year, 8 quarters = the two-year decline the EDA found.
HORIZONS = [4, 8]

print(f"{len(RATIO_TREND)} ratios + {len(GROWTH_TREND)} levels, at {HORIZONS} quarters back")
print(f"-> {(len(RATIO_TREND) + len(GROWTH_TREND)) * len(HORIZONS)} new columns")

16 ratios + 2 levels, at [4, 8] quarters back
-> 36 new columns


## 2 · Look up the past by date, not by row position

The obvious way to get "a year ago" is `shift(4)`, go back four rows within each bank.
That is wrong whenever a bank has a missing quarter: the shift silently returns a filing
from five or six quarters back and labels it as one year. Only 17 rows in this panel sit
after such a gap, but a wrong lag is invisible once it is in the table, so it is worth
closing off entirely.

Instead, each row gets a **quarter number** (`year × 4 + quarter`), and the past is fetched
by joining the table to itself on `(bank, quarter number − 4)`. If the filing does not
exist, the result is blank, which is honest, rather than the wrong quarter.

**On leakage:** every lookup goes *backward*. Nothing here reads a quarter later than the
row it is attached to, so these columns are safe under the guardrail in `modeling.ipynb`.

In [3]:
panel["_qi"] = panel["REPDTE"].dt.year * 4 + panel["REPDTE"].dt.quarter

# Confirm the join key is unique before relying on it.
assert not panel.duplicated(["CERT", "_qi"]).any(), "a bank has two filings in one quarter"

base = panel[["CERT", "_qi"] + RATIO_TREND + GROWTH_TREND]

for h in HORIZONS:
    # Shift the *past* forward by h quarters so it lines up with the row it belongs to.
    past = base.copy()
    past["_qi"] = past["_qi"] + h
    past = past.rename(columns={c: f"{c}__prev" for c in RATIO_TREND + GROWTH_TREND})

    panel = panel.merge(past, on=["CERT", "_qi"], how="left")

    for c in RATIO_TREND:
        panel[f"{c}_chg{h}q"] = panel[c] - panel[f"{c}__prev"]

    for c in GROWTH_TREND:
        # Guard against a zero denominator producing an infinite growth rate.
        prior = panel[f"{c}__prev"].replace(0, np.nan)
        panel[f"{c}_grow{h}q"] = panel[c] / prior - 1

    panel = panel.drop(columns=[f"{c}__prev" for c in RATIO_TREND + GROWTH_TREND])

panel = panel.drop(columns=["_qi"])

TREND_COLS = [c for c in panel.columns if "_chg" in c or "_grow" in c]
print(f"added {len(TREND_COLS)} trend columns -> {panel.shape[1]} total")

added 36 trend columns -> 103 total


## 3 · How complete are they

A trend column is blank whenever the bank has no filing that far back, a young bank, or
one that joined the panel recently. That is a real gap, not an error, and it should stay
blank rather than be filled with a zero, which would read as "no change" and quietly claim
the bank was stable when nothing is known about it.

The 8-quarter columns are emptier than the 4-quarter ones for the same reason, and the
funding columns inherit the era gaps already documented in `cleaning.ipynb`
(`DEPUNA` starts ~1993).

In [4]:
gaps = (panel[TREND_COLS].isna().mean() * 100).sort_values(ascending=False)

print(f"missing %:  min {gaps.min():.1f}   median {gaps.median():.1f}   max {gaps.max():.1f}\n")
print("emptiest five:")
print(gaps.head().round(1).to_string())
print("\nfullest five:")
print(gaps.tail().round(1).to_string())

missing %:  min 6.0   median 11.6   max 30.1

emptiest five:
DEPUNA_chg8q     30.1
DEPUNA_chg4q     24.1
RBCRWAJ_chg8q    15.6
DEP_grow8q       12.3
ROE_chg8q        11.9

fullest five:
EQV_chg4q         6.0
ROA_chg4q         6.0
LNATRESR_chg4q    6.0
NTLNLSR_chg4q     6.0
NCLNLSR_chg4q     6.0


## 4 · Do the trends actually carry signal

Before trusting these columns, check each one alone: how well does it separate "becomes
undercapitalized within a year" from "stays fine"? Scored by AUC, where 0.5 is a coin flip.

**Computed on the training years only (1990–2015).** Ranking features on the full panel
would let the test period influence which features get kept, the feature-selection
leakage noted at the end of `modeling.ipynb`.

In [5]:
train_rows = panel[(panel["onset_4q"].notna()) & (panel["REPDTE"] <= "2015-12-31")]


def solo_auc(col: str) -> tuple[float, int]:
    """AUC of one column on its own. Direction-free: a feature that predicts perfectly
    backwards is just as informative as one that predicts perfectly forwards."""
    s = train_rows[[col, "onset_4q"]].dropna()
    if s["onset_4q"].nunique() < 2:
        return float("nan"), len(s)
    a = roc_auc_score(s["onset_4q"], s[col])
    return max(a, 1 - a), len(s)


scores = pd.DataFrame(
    [(c, *solo_auc(c)) for c in TREND_COLS], columns=["feature", "auc", "n"]
).sort_values("auc", ascending=False)

# The capital ratio level, for reference: the bar every trend feature is measured against.
level_auc, _ = solo_auc("RBCRWAJ")

print(f"capital ratio LEVEL (reference): {level_auc:.3f}\n")
print(scores.head(12).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

capital ratio LEVEL (reference): 0.879

       feature   auc      n
  NPERFV_chg8q 0.790 893034
 NCLNLSR_chg8q 0.769 893034
     ROE_chg8q 0.755 892938
 RBC1AAJ_chg8q 0.749 893034
  NPERFV_chg4q 0.745 959204
     ROA_chg8q 0.740 893034
LNATRESR_chg8q 0.733 893034
    ORER_chg8q 0.733 893034
     EQV_chg8q 0.729 893034
 RBC1AAJ_chg4q 0.719 959204
 NTLNLSR_chg8q 0.717 893034
 NCLNLSR_chg4q 0.704 959204


### What the scores say

The trends carry real signal on their own, change in bad loans and change in capital both
land well above a coin flip, but **none of them beats the capital ratio's level**.

That is the expected result, and it is worth stating plainly rather than glossing: the
target is defined by capital crossing a threshold, so the current level sits mechanically
close to the answer. A bank at 9% is one bad quarter from 8%.

The trends earn their place only if they add something the level does not already containwhich is a question about the *combination*, not about any single column, and so cannot be
answered here. It gets answered in `modeling.ipynb`, by whether the model beats the
capital-ratio benchmark.

## 5 · The funding signals the capital ratios cannot see

Everything above is a trend on a ratio the regulators already watch. But the EDA case study
showed the limit of that: Silicon Valley Bank's capital ratio *rose* from 11.5% to 16% on
the way to failure, and its nonperforming assets stayed near zero. Nothing in the classic
CAMELS ratios moved.

What did move sat in columns nobody was reading, the bond book and the deposit base. Three
measures, each already in the panel as raw fields:

| Feature | Built from | What it catches |
|---|---|---|
| `htm_loss_pct` | `SCHA` − `SCHF`, over assets | Bonds the bank holds at cost that are worth less than that. The loss the capital ratio never books |
| `afs_loss_pct` | `SCAA` − `SCAF`, over assets | The same for the tradeable side of the portfolio |
| `uninsured_pct` | `DEPUNA` / `DEP` | The share of deposits above the insurance limit, the money that runs first |

SVB's held-to-maturity losses reached 7.6% of assets against roughly zero for the median
bank over $50B, and uninsured deposits stood at 86%. Both were visible in the filings.

**Two field traps here, both found the hard way.** `ESTINS` looks like it should be insured
deposits in dollars, but it is a *percentage*, using it as a dollar amount makes every bank
look 100% uninsured. And `DEPUNA` uses **0 as a non-report placeholder**: 62-69% of banks
under $1B report exactly zero uninsured deposits, which is not possible, while banks that do
report show a median of 12-33%. Same lesson as the CBLR zero-fill in `data_concerns.md`, a
zero and a blank are different lies. Both are handled below.

Each gets the same 4- and 8-quarter change treatment as the ratios above, since it was the
*trend* that told the story, not any single quarter's level.

In [6]:
# Amortized cost minus fair value = the unrealized loss sitting in the bond book.
panel["htm_loss_pct"] = (panel["SCHA"] - panel["SCHF"]) / panel["ASSET"] * 100
panel["afs_loss_pct"] = (panel["SCAA"] - panel["SCAF"]) / panel["ASSET"] * 100

# Deposits above the FDIC insurance limit, as a share of all deposits.
# DEPUNA is the dollar amount. (ESTINS is a percentage, not dollars -- do not subtract it.)
uninsured = panel["DEPUNA"] / panel["DEP"].replace(0, np.nan) * 100

# A zero in DEPUNA means "not reported", not "no uninsured deposits" -- no real bank has
# none. Blank it rather than let it read as a bank with the safest possible funding.
uninsured = uninsured.where(panel["DEPUNA"] > 0)

# 31 rows report more uninsured deposits than total deposits. Impossible; blank them.
panel["uninsured_pct"] = uninsured.where(uninsured <= 100)

FUNDING = ["htm_loss_pct", "afs_loss_pct", "uninsured_pct"]

# Same backward-only lookup as before.
panel["_qi"] = panel["REPDTE"].dt.year * 4 + panel["REPDTE"].dt.quarter

for h in HORIZONS:
    past = panel[["CERT", "_qi"] + FUNDING].copy()
    past["_qi"] = past["_qi"] + h
    past = past.rename(columns={c: f"{c}__prev" for c in FUNDING})

    panel = panel.merge(past, on=["CERT", "_qi"], how="left")
    for c in FUNDING:
        panel[f"{c}_chg{h}q"] = panel[c] - panel[f"{c}__prev"]
    panel = panel.drop(columns=[f"{c}__prev" for c in FUNDING])

panel = panel.drop(columns=["_qi"])

FUNDING_COLS = FUNDING + [f"{c}_chg{h}q" for c in FUNDING for h in HORIZONS]
TREND_COLS = TREND_COLS + [c for c in FUNDING_COLS if c not in TREND_COLS]

print(f"added {len(FUNDING_COLS)} funding columns -> {panel.shape[1]} total")

added 9 funding columns -> 112 total


### Does the SVB story hold up in the numbers?

Two checks. First, the same single-feature AUC as above. Second, more to the point, what
these columns looked like for Silicon Valley Bank in its final quarters, against the median
bank of similar size.

In [7]:
# train_rows was sliced before these columns existed -- refresh it.
train_rows = panel[(panel["onset_4q"].notna()) & (panel["REPDTE"] <= "2015-12-31")]

print("single-feature AUC, training years only:\n")
fund_scores = pd.DataFrame(
    [(c, *solo_auc(c)) for c in FUNDING_COLS], columns=["feature", "auc", "n"]
).sort_values("auc", ascending=False)
print(fund_scores.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# SVB (CERT 24735) in the two years before failure, against same-size peers.
svb = panel[(panel["CERT"] == 24735) & (panel["REPDTE"] >= "2021-06-30")]

if len(svb):
    big = panel[(panel["ASSET"] > 5e7) & (panel["REPDTE"].isin(svb["REPDTE"]))]
    print("\n\nSilicon Valley Bank vs the median bank over $50B, final quarters:\n")
    view = svb[["REPDTE", "RBCRWAJ", "htm_loss_pct", "uninsured_pct"]].copy()
    view["peer_htm"] = view["REPDTE"].map(big.groupby("REPDTE")["htm_loss_pct"].median())
    view["peer_uninsured"] = view["REPDTE"].map(big.groupby("REPDTE")["uninsured_pct"].median())
    view["REPDTE"] = view["REPDTE"].dt.date
    print(view.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
    print("\nCapital ratio rises while the bond losses and uninsured share do not.")

single-feature AUC, training years only:



            feature   auc      n
uninsured_pct_chg8q 0.687 137192
uninsured_pct_chg4q 0.649 178981
      uninsured_pct 0.552 225777
       afs_loss_pct 0.542 733319
       htm_loss_pct 0.540 733319
 htm_loss_pct_chg8q 0.520 619947
 afs_loss_pct_chg8q 0.519 619947
 htm_loss_pct_chg4q 0.513 675099
 afs_loss_pct_chg4q 0.501 675099


Silicon Valley Bank vs the median bank over $50B, final quarters:

    REPDTE  RBCRWAJ  htm_loss_pct  uninsured_pct  peer_htm  peer_uninsured
2021-06-30     14.3          -0.1           86.4      -0.0            47.6
2021-09-30     15.2           0.2           86.9      -0.0            48.2
2021-12-31     15.4           0.5           86.7       0.0            49.9
2022-03-31     15.4           3.2           86.3       0.1            47.6
2022-06-30     16.0           5.3           86.9       0.3            48.5
2022-09-30     16.2           7.6           86.8       0.6            48.1
2022-12-31     16.1           7.3           86.4       0.4            48.5



## 6 · Reaching for yield

Carmona, Climent & Momparler (2018), the one paper in the review using gradient boosting on
U.S. bank failure, report a predictor the other papers miss: an **exceedingly high yield on
earning assets** raises failure risk.

The logic is that a bank earning unusually high yields is not clever, it is taking more
credit or duration risk than its peers to do it. High yield is a symptom, not a strength.

Two fields, both Carmona's:

| Feature | FDIC field | Meaning |
|---|---|---|
| `INTINCY` | Yield on earning assets | What the bank earns on what it lends and holds |
| `INTEXPY` | Cost of funding earning assets | What it pays for the money to do that |

Neither is in the panel built by `feature_selection.ipynb`, so they are pulled straight from
the raw quarterly files and joined on bank and quarter.

**Yield is only meaningful against the market.** A 6% yield was unremarkable in 1995 and
extraordinary in 2021, so the raw level would mostly encode which decade a row is from. Each
is therefore also expressed as a **spread over the median bank that same quarter**, how far
above the pack this bank was reaching, at that moment.

In [8]:
import glob

RAW = Path("..") / "data" / "raw"

yield_cols = pd.concat(
    [pd.read_parquet(f, columns=["CERT", "REPDTE", "INTINCY", "INTEXPY"])
     for f in sorted(glob.glob(str(RAW / "financials_*.parquet")))],
    ignore_index=True,
)
yield_cols["REPDTE"] = pd.to_datetime(yield_cols["REPDTE"], format="%Y%m%d")

panel = panel.merge(yield_cols, on=["CERT", "REPDTE"], how="left")

# Spread over the median bank in the same quarter -- strips out the rate cycle, so what
# is left is how far above its peers this bank was reaching.
for col, name in [("INTINCY", "yield_spread"), ("INTEXPY", "funding_cost_spread")]:
    panel[name] = panel[col] - panel.groupby("REPDTE")[col].transform("median")

YIELD_BASE = ["INTINCY", "INTEXPY", "yield_spread", "funding_cost_spread"]

# Same backward-only change treatment as everything else.
panel["_qi"] = panel["REPDTE"].dt.year * 4 + panel["REPDTE"].dt.quarter
for h in HORIZONS:
    past = panel[["CERT", "_qi"] + YIELD_BASE].copy()
    past["_qi"] = past["_qi"] + h
    past = past.rename(columns={c: f"{c}__prev" for c in YIELD_BASE})
    panel = panel.merge(past, on=["CERT", "_qi"], how="left")
    for c in YIELD_BASE:
        panel[f"{c}_chg{h}q"] = panel[c] - panel[f"{c}__prev"]
    panel = panel.drop(columns=[f"{c}__prev" for c in YIELD_BASE])
panel = panel.drop(columns=["_qi"])

YIELD_COLS = YIELD_BASE + [f"{c}_chg{h}q" for c in YIELD_BASE for h in HORIZONS]
TREND_COLS = TREND_COLS + [c for c in YIELD_COLS if c not in TREND_COLS]

print(f"added {len(YIELD_COLS)} yield columns -> {panel.shape[1]} total")
print(f"INTINCY missing: {panel['INTINCY'].isna().mean() * 100:.1f}%")

added 12 yield columns -> 124 total
INTINCY missing: 0.0%


### Does reaching for yield show up?

Same single-feature check as before. Carmona's claim is directional, *high* yield means
higher risk, so it is worth seeing whether the spread version carries more than the raw
level, which would confirm that the signal is peer-relative rather than a rate-cycle artefact.

In [9]:
train_rows = panel[(panel["onset_4q"].notna()) & (panel["REPDTE"] <= "2015-12-31")]

yield_scores = pd.DataFrame(
    [(c, *solo_auc(c)) for c in YIELD_COLS], columns=["feature", "auc", "n"]
).sort_values("auc", ascending=False)
print(yield_scores.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# Direction check: do banks heading for distress sit above or below their peers?
flag = train_rows["onset_4q"] == 1
print(f"\nmedian yield spread, banks that later became undercapitalized: "
      f"{train_rows.loc[flag, 'yield_spread'].median():+.3f} pts")
print(f"median yield spread, banks that stayed safe:                    "
      f"{train_rows.loc[~flag, 'yield_spread'].median():+.3f} pts")

                  feature   auc       n
      funding_cost_spread 0.712 1026797
                  INTEXPY 0.661 1026797
            INTINCY_chg4q 0.642  959204
       yield_spread_chg8q 0.622  893034
            INTINCY_chg8q 0.614  893034
       yield_spread_chg4q 0.613  959204
                  INTINCY 0.604 1026797
             yield_spread 0.600 1026797
            INTEXPY_chg4q 0.590  959204
funding_cost_spread_chg8q 0.550  893034
funding_cost_spread_chg4q 0.520  959204
            INTEXPY_chg8q 0.518  893034

median yield spread, banks that later became undercapitalized: +0.252 pts
median yield spread, banks that stayed safe:                    +0.000 pts


## 7 · Save

No rows dropped, no existing value altered, 57 columns added.
`modeling.ipynb` reads this file.

In [10]:
out = PROCESSED / "panel_trend.parquet"
panel.to_parquet(out, index=False)

print(f"saved {out.name}: {panel.shape[0]:,} rows x {panel.shape[1]} cols")
print(f"  {len(TREND_COLS)} trend columns added")

saved panel_trend.parquet: 1,258,888 rows x 124 cols
  57 trend columns added
